# 03 - LoRA Fine-Tuning: Climate-Aware Crop Disease Advisory Assistant

This notebook fine-tunes a small instruct LLM with **LoRA** and runs the experiments **E2-E5** from the project plan, plus the zero-shot baselines **B2/E1**.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** (free tier is enough for a 1.5B model).
2. Put the project folder on Google Drive (`MyDrive/crop-advisory-assistant/`) containing `src/prompting.py`, `src/train_lora.py`, and `data/processed/{train,val,test}.jsonl`.
3. Every `jsonl` row follows the schema documented at the top of `src/prompting.py` (`question`, `answer`, optional `context`, `climate`, `source_doc`).

**What LoRA does (be ready to explain this in your demo):** instead of updating a full weight matrix `W`, we freeze `W` and learn a low-rank update `ΔW = (alpha / r) · B·A`, where `A` is `r × k` and `B` is `d × r`. Only `A` and `B` are trained, so trainable parameters drop from millions per matrix to a small fraction, while the frozen base model keeps its general language ability (Hu et al., 2021).

## 1. Environment

In [ ]:
!nvidia-smi
!pip install -q -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "datasets" "bitsandbytes" evaluate rouge_score bert_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT = '/content/drive/MyDrive/crop-advisory-assistant'   # <- change if your folder is elsewhere
os.chdir(PROJECT)
sys.path.insert(0, f'{PROJECT}/src')
print(os.listdir('.'))

## 2. Data check

If you have not built your real dataset yet, the next cell creates a **tiny smoke-test dataset** so you can verify the whole pipeline runs end to end. These rows exist only to test the code; they are **not** a real dataset and must not be used for results. Delete the cell's output files once your real data is ready.

In [ ]:
import json, os
os.makedirs('data/processed', exist_ok=True)

if not os.path.exists('data/processed/train.jsonl'):
    base = [
        {"question": "What weather favours potato late blight?",
         "answer": "Late blight is favoured by cool, wet and humid conditions. Monitor fields closely after several days of rain and high humidity, and contact your extension officer for control options.",
         "context": ["Potato late blight spreads fastest in cool temperatures with high humidity or prolonged leaf wetness."],
         "climate": {"temperature_c": 16, "humidity_pct": 92, "rainfall_mm_7d": 70, "season": "rainy"}},
        {"question": "What weather favours potato late blight?",
         "answer": "Hot, dry weather is less favourable for late blight, so risk is lower now. Keep scouting, because conditions can change quickly when rain returns.",
         "context": ["Potato late blight spreads fastest in cool temperatures with high humidity or prolonged leaf wetness."],
         "climate": {"temperature_c": 30, "humidity_pct": 40, "rainfall_mm_7d": 0, "season": "dry"}},
        {"question": "How can I reduce the spread of cassava mosaic disease?",
         "answer": "Use clean, certified or disease-free cuttings, remove and destroy infected plants early, and choose resistant varieties where available.",
         "context": ["Cassava mosaic disease is spread through infected cuttings and by whiteflies. Clean planting material and resistant varieties reduce losses."],
         "climate": {"temperature_c": 25, "humidity_pct": 65, "rainfall_mm_7d": 15, "season": "dry"}},
        {"question": "Who won the football match yesterday?",
         "answer": "I can only help with crop disease questions for maize, beans, cassava and potato. Please ask me about those crops.",
         "context": [], "climate": None},
    ]
    def dump(name, rows, reps):
        with open(f'data/processed/{name}.jsonl', 'w') as f:
            for r in range(reps):
                for i, ex in enumerate(rows):
                    f.write(json.dumps({**ex, "id": f"{name}-{r}-{i}", "source_doc": f"{name}-doc-{r}"}) + "\n")
    dump('train', base, 8); dump('val', base, 2); dump('test', base, 2)
    print('Created SMOKE-TEST data only.')

for s in ['train', 'val', 'test']:
    print(s, sum(1 for _ in open(f'data/processed/{s}.jsonl')))

**Leakage check.** The brief and the plan both require splitting by source document. This cell verifies that no `source_doc` appears in more than one split.

In [ ]:
def read_jsonl(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
splits = {s: read_jsonl(f'data/processed/{s}.jsonl') for s in ['train', 'val', 'test']}
docs = {s: {r.get('source_doc') for r in rows} for s, rows in splits.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = docs[a] & docs[b]
    print(f'{a} vs {b}: {len(overlap)} shared source docs', '(OK)' if not overlap else '<-- LEAKAGE, fix your split')

## 3. Look at one training prompt

Always inspect the exact text the model is trained on. Note that **only the answer** contributes to the loss.

In [ ]:
from prompting import build_messages
from transformers import AutoTokenizer
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
ex = splits['train'][0]
print(tok.apply_chat_template(build_messages(ex, use_context=True, use_climate=True), tokenize=False, add_generation_prompt=True))
print('--- TARGET ---')
print(ex['answer'])

## 4. Run the experiments

Each run calls `src/train_lora.py` in a subprocess so GPU memory is fully released between runs.

| Run | Prompt contains | Plan ID |
|---|---|---|
| `e2_lora_noctx` | question only | E2 |
| `e3_lora_rag` | question + passages | E3 |
| `e4_lora_rag_climate` | question + passages + weather | E4 |
| `e5_*` | same as E3 or E4 (choose the best), varying rank / learning rate | E5 |

If you run out of memory: add `--load_in_4bit` (QLoRA), lower `--batch_size` to 2 and raise `--grad_accum`, or lower `--max_len`.

In [ ]:
import subprocess, shlex

def run_training(name, use_context=False, use_climate=False, r=16, alpha=None, lr=2e-4, epochs=3,
                 batch_size=4, grad_accum=4, extra=''):
    alpha = alpha or 2 * r
    cmd = (f'python src/train_lora.py --train_file data/processed/train.jsonl --val_file data/processed/val.jsonl '
           f'--output_dir results/runs/{name} --r {r} --lora_alpha {alpha} --lr {lr} --epochs {epochs} '
           f'--batch_size {batch_size} --grad_accum {grad_accum} {extra}')
    if use_context: cmd += ' --use_context'
    if use_climate: cmd += ' --use_climate'
    print('>>>', cmd)
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    assert proc.returncode == 0, f'{name} failed'

# Start with ONE run to confirm everything works, then uncomment the others.
run_training('e3_lora_rag', use_context=True)
# run_training('e2_lora_noctx')
# run_training('e4_lora_rag_climate', use_context=True, use_climate=True)

### E5: hyper-parameter study (run after E2-E4; apply to your best configuration)

In [ ]:
# for r, lr in [(8, 1e-4), (8, 2e-4), (16, 1e-4), (16, 2e-4)]:
#     run_training(f'e5_r{r}_lr{lr}', use_context=True, use_climate=True, r=r, lr=lr)

## 5. Training results and loss curves

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt

rows = []
for path in sorted(glob.glob('results/runs/*/metrics.json')):
    m = json.load(open(path)); a = m['args']
    rows.append({'run': path.split('/')[-2], 'context': a['use_context'], 'climate': a['use_climate'],
                 'r': a['r'], 'lr': a['lr'], 'trainable_params': m['trainable_params'],
                 'val_loss': round(m['eval_loss'], 4), 'val_ppl': round(m['eval_perplexity'], 2),
                 'minutes': m['train_minutes']})
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv('results/training_summary.csv', index=False)

plt.figure(figsize=(7, 4))
for path in sorted(glob.glob('results/runs/*/metrics.json')):
    log = json.load(open(path))['log_history']
    steps = [x['step'] for x in log if 'loss' in x]; loss = [x['loss'] for x in log if 'loss' in x]
    plt.plot(steps, loss, label=path.split('/')[-2])
plt.xlabel('step'); plt.ylabel('training loss'); plt.legend(); plt.title('Training loss'); plt.tight_layout()
plt.savefig('results/loss_curves.png', dpi=150); plt.show()

> **Caution:** validation loss/perplexity are only comparable between runs with the **same prompt format** (e.g. E3 vs its r/lr variants). A prompt with passages has different tokens to predict conditioned on different input, so compare E2/E3/E4 using the generation metrics below, not just loss.

## 6. Generate answers on the test set and score them

We compare **B2** (base model, adapter off, no passages), **E1** (base model, adapter off, with passages) and the fine-tuned adapters. The adapter is toggled with `disable_adapter()`, so the base model is loaded once.

In [ ]:
import torch, evaluate
from transformers import AutoModelForCausalLM
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token

def load_with_adapter(adapter_dir):
    base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map={'': 0})
    return PeftModel.from_pretrained(base, adapter_dir).eval()

@torch.no_grad()
def generate(model, examples, use_context, use_climate, adapter=True, batch_size=8, max_new_tokens=256):
    outs = []
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        prompts = [tok.apply_chat_template(build_messages(e, use_context=use_context, use_climate=use_climate),
                                           tokenize=False, add_generation_prompt=True) for e in batch]
        enc = tok(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
        gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05,
                          pad_token_id=tok.pad_token_id)
        if adapter:
            ids = model.generate(**enc, **gen_kwargs)
        else:
            with model.disable_adapter():
                ids = model.generate(**enc, **gen_kwargs)
        outs += tok.batch_decode(ids[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

rouge = evaluate.load('rouge')
def score(preds, refs):
    r = rouge.compute(predictions=preds, references=refs)
    return {'rougeL': round(r['rougeL'], 4), 'rouge1': round(r['rouge1'], 4)}

test = splits['test']
refs = [e['answer'] for e in test]
results, all_preds = [], {}

In [ ]:
# Use one adapter just to host the base model; adapter=False bypasses LoRA entirely.
model = load_with_adapter('results/runs/e3_lora_rag/adapter')

for name, ctx, clim in [('B2_base_zero_shot', False, False), ('E1_base_rag', True, False)]:
    preds = generate(model, test, ctx, clim, adapter=False)
    all_preds[name] = preds; results.append({'system': name, **score(preds, refs)})
display(pd.DataFrame(results))

In [ ]:
# Fine-tuned systems: (run name, use_context, use_climate) must match how each run was trained.
FINETUNED = [
    ('e3_lora_rag', True, False),
    # ('e2_lora_noctx', False, False),
    # ('e4_lora_rag_climate', True, True),
]
del model; torch.cuda.empty_cache()
for run, ctx, clim in FINETUNED:
    m = load_with_adapter(f'results/runs/{run}/adapter')
    preds = generate(m, test, ctx, clim, adapter=True)
    all_preds[run] = preds; results.append({'system': run, **score(preds, refs)})
    del m; torch.cuda.empty_cache()

table = pd.DataFrame(results); display(table)
table.to_csv('results/generation_scores.csv', index=False)
with open('results/predictions.jsonl', 'w') as f:
    for i, e in enumerate(test):
        f.write(json.dumps({'id': e.get('id'), 'question': e['question'], 'reference': e['answer'],
                            **{k: v[i] for k, v in all_preds.items()}}) + '\n')

Optional: BERTScore (semantic similarity). It downloads an extra model; use `lang='en'`.

In [ ]:
# from bert_score import score as bert_score
# for name, preds in all_preds.items():
#     P, R, F1 = bert_score(preds, refs, lang='en', verbose=False)
#     print(name, round(F1.mean().item(), 4))

## 7. Climate-sensitivity test (E4)

Same question + same passages, **different weather**. If the model is truly climate-aware, the advice should change appropriately. Inspect the outputs yourself and include them in the report's error analysis. Use the E4 adapter (trained with `--use_climate`).

In [ ]:
probe_q = 'Should I be worried about late blight in my potato field this week?'
passages = ['Potato late blight spreads fastest in cool temperatures with high humidity or prolonged leaf wetness.']
wet = {'question': probe_q, 'context': passages,
       'climate': {'temperature_c': 16, 'humidity_pct': 92, 'rainfall_mm_7d': 70, 'season': 'rainy'}}
dry = {'question': probe_q, 'context': passages,
       'climate': {'temperature_c': 30, 'humidity_pct': 40, 'rainfall_mm_7d': 0, 'season': 'dry'}}

run = 'e4_lora_rag_climate'          # change if you named it differently
if os.path.exists(f'results/runs/{run}/adapter'):
    m = load_with_adapter(f'results/runs/{run}/adapter')
    for label, ex in [('COOL + WET', wet), ('HOT + DRY', dry)]:
        print(f'[{label}]\n', generate(m, [ex], True, True)[0], '\n')
    del m; torch.cuda.empty_cache()
else:
    print('Train the E4 run first.')

## 8. Out-of-domain behaviour

The assistant should decline off-topic questions. Check this qualitatively here, and quantitatively with the domain-gate classifier (a separate notebook).

In [ ]:
ood = [{'question': q, 'context': [], 'climate': None} for q in [
    'Who is the president of France?', 'How do I treat malaria?', 'Write a poem about the sea.']]
m = load_with_adapter('results/runs/e3_lora_rag/adapter')
for e, out in zip(ood, generate(m, ood, True, False)):
    print('Q:', e['question'], '\nA:', out, '\n')
del m; torch.cuda.empty_cache()

## 9. Merge the adapter and (optionally) push to the Hugging Face Hub

The merged model is what the Gradio app on Hugging Face Spaces will load. For a free CPU Space you may need a smaller model or a quantized (GGUF) export; decide this when you reach deployment.

In [ ]:
# Merge (writes results/runs/<run>/merged):
# (Alternatively, pass --merge to train_lora.py during training.)
from train_lora import merge_adapter
BEST = 'e3_lora_rag'   # <- your best run
merge_adapter(BASE, f'results/runs/{BEST}/adapter', f'results/runs/{BEST}/merged')

# from huggingface_hub import notebook_login; notebook_login()
# from transformers import AutoModelForCausalLM
# AutoModelForCausalLM.from_pretrained(f'results/runs/{BEST}/merged').push_to_hub('YOUR_USERNAME/crop-advisory-qwen-lora')
# tok.push_to_hub('YOUR_USERNAME/crop-advisory-qwen-lora')

## 10. Things to be able to explain in your demo

- Why only answer tokens are in the loss (label masking with `-100`).
- What `r` and `alpha` control (capacity of the update, and its scaling `alpha / r`), and why LoRA targets the attention and MLP projections.
- Why we use fp16/bf16, gradient checkpointing and gradient accumulation (memory vs effective batch size = `batch_size x grad_accum`).
- Why early stopping on validation loss, and why splits are by source document.
- Why ROUGE-L alone is not enough for advice quality, and what your human evaluation adds.
- If E3 does not beat E1 or B2, what your evidence says about why (data size, synthetic answers, retrieval quality, model capacity).